In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, average_precision_score, f1_score, 
    precision_recall_curve, roc_curve, roc_auc_score,
    accuracy_score, precision_score, recall_score, confusion_matrix
)
from codecarbon import EmissionsTracker
import re

# ====== CONFIG ======
IN_FILE = "risultati_merged_mixed_enriched.csv"
MODEL_OUT = "catboost_td_full.cbm"
META_OUT = "catboost_td_full_meta.json"
PLOTS_DIR = "plots_catboost_full"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"
LABEL_COL = "label"
RANDOM_STATE = 42
TEST_SIZE = 0.2
# Oversampling "soft" dei borderline (vicini a 0)
USE_BORDERLINE_WEIGHTS = True
W_MAX = 5.0  # peso massimo vicino a logit=0

# Crea directory per i grafici
os.makedirs(PLOTS_DIR, exist_ok=True)

# ====== LOAD ======
print("Caricamento dataset...")
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

print(f"Dataset caricato: {len(df)} righe")

# ====== CHEAP FEATURES ======
print("Estrazione cheap features...")
KW = ["todo", "fixme", "hack", "workaround", "temp", "quick", "debt", "refactor"]
_re_word = re.compile(r"\b\w+\b", re.UNICODE)
_re_upper = re.compile(r"[A-Z]")
_re_digit = re.compile(r"\d")
_re_punct = re.compile(r"[^\w\s]")
_re_snake = re.compile(r"\b[a-z]+_[a-z0-9_]+\b")
_re_camel = re.compile(r"\b[a-z]+[A-Z][A-Za-z0-9]*\b")

def kw_count_word(tl: str, kw: str) -> int:
    return len(re.findall(rf"\b{re.escape(kw)}\b", tl))

def cheap_features(text: str) -> dict:
    t = "" if text is None else str(text)
    tl = t.lower()
    n_chars = len(t)
    words = _re_word.findall(t)
    n_words = len(words) if words else 0
    n_upper = len(_re_upper.findall(t))
    n_digit = len(_re_digit.findall(t))
    n_punct = len(_re_punct.findall(t))
    pct_upper = n_upper / n_chars if n_chars else 0.0
    pct_digit = n_digit / n_chars if n_chars else 0.0
    pct_punct = n_punct / n_chars if n_chars else 0.0
    
    feats = {
        "n_chars": n_chars,
        "n_words": n_words,
        "pct_upper": pct_upper,
        "pct_digit": pct_digit,
        "pct_punct": pct_punct,
        "has_should": int(" should " in f" {tl} "),
        "has_later": int(" later " in f" {tl} "),
        "has_for_now": int(" for now " in tl),
        "has_snake_case": int(_re_snake.search(t) is not None),
        "has_camel_case": int(_re_camel.search(t) is not None),
        "has_path": int(("/" in t) or ("\\" in t)),
        "has_equals": int("=" in t),
        "has_brackets": int(any(ch in t for ch in "(){}[]")),
    }
    
    for k in KW:
        cnt = kw_count_word(tl, k)
        feats[f"kw_{k}_cnt"] = cnt
        feats[f"kw_{k}_has"] = int(cnt > 0)
    
    return feats

df_feats = df.copy()
cheap = df_feats[TEXT_COL].apply(cheap_features).apply(pd.Series)
df_feats = pd.concat([df_feats, cheap], axis=1)

# Feature columns: TESTO + tutte le numeriche
num_cols = [c for c in df_feats.columns if c.startswith(("n_", "pct_", "kw_", "has_"))]
feature_cols = [TEXT_COL] + num_cols

print(f"Features estratte: {len(feature_cols)} (1 testo + {len(num_cols)} numeriche)")

# Gestione NaN/Inf
for col in num_cols:
    df_feats[col] = df_feats[col].replace([np.inf, -np.inf], np.nan)
    df_feats[col] = df_feats[col].fillna(0.0)

# ====== SPLIT ======
print("Split train/validation...")
strat = df_feats[LABEL_COL] if (LABEL_COL in df_feats.columns and df_feats[LABEL_COL].notna().any()) else None
train_df, valid_df = train_test_split(
    df_feats,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=strat
)

print(f"Train: {len(train_df)} | Validation: {len(valid_df)}")

# ====== WEIGHTS (borderline focus) ======
train_w = None
if USE_BORDERLINE_WEIGHTS:
    a = np.abs(train_df[TARGET_COL].values)
    # peso continuo: vicino a 0 -> ~W_MAX, lontano -> ~1
    train_w = 1.0 + (W_MAX - 1.0) * np.exp(-a)
    print(f"Borderline weighting attivo: W_MAX={W_MAX}, peso medio={train_w.mean():.3f}")

# ====== POOLS ======
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL],
    weight=train_w
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN CON CARBON TRACKING ======
print("\nInizio training con carbon tracking...")
tracker = EmissionsTracker(project_name="catboost_full", output_dir=PLOTS_DIR)
tracker.start()

start_time = time.time()

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=6000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

training_time = time.time() - start_time
emissions = tracker.stop()

print(f"\nTraining completato in {training_time:.2f} secondi ({training_time/60:.2f} minuti)")
print(f"Emissioni CO2: {emissions:.6f} kg")

# ====== PREDICTIONS ======
print("\nGenerazione predizioni...")
pred_logit_train = model.predict(train_pool)
pred_logit = model.predict(valid_pool)

# ====== EVAL (distillation fidelity) ======
rmse_train = np.sqrt(mean_squared_error(train_df[TARGET_COL].values, pred_logit_train))
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) train: {rmse_train:.6f}")
print(f"RMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (classification vs label) ======
p_hat_train = 1.0 / (1.0 + np.exp(-pred_logit_train))
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))

metrics = {}

if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_train = train_df[LABEL_COL].astype(int).values
    y_true = valid_df[LABEL_COL].astype(int).values
    
    # ROC-AUC
    roc_auc_train = roc_auc_score(y_train, p_hat_train)
    roc_auc = roc_auc_score(y_true, p_hat)
    print(f"\nROC-AUC train: {roc_auc_train:.6f}")
    print(f"ROC-AUC valid: {roc_auc:.6f}")
    
    # PR-AUC
    pr_auc_train = average_precision_score(y_train, p_hat_train)
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC train: {pr_auc_train:.6f}")
    print(f"PR-AUC valid: {pr_auc:.6f}")
    
    # Trova soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    
    y_pred = (p_hat >= best_thr).astype(int)
    
    # Metriche di classificazione
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\nSoglia ottimale (F1): {best_thr:.6f}")
    print(f"Accuracy: {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall: {recall:.6f}")
    print(f"F1-Score: {f1:.6f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    metrics = {
        "roc_auc_train": roc_auc_train,
        "roc_auc_valid": roc_auc,
        "pr_auc_train": pr_auc_train,
        "pr_auc_valid": pr_auc,
        "best_threshold": best_thr,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }
    
    # ====== PLOT ROC CURVE ======
    fpr, tpr, _ = roc_curve(y_true, p_hat)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve - CatBoost Full Model', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "roc_curve.png"), dpi=300)
    print(f"\nROC curve salvata in: {os.path.join(PLOTS_DIR, 'roc_curve.png')}")
    plt.close()
    
    # ====== PLOT PR CURVE ======
    plt.figure(figsize=(8, 6))
    plt.plot(recalls, precisions, linewidth=2, label=f'PR curve (AUC = {pr_auc:.4f})')
    plt.axhline(y=y_true.mean(), color='k', linestyle='--', linewidth=1, label=f'Baseline ({y_true.mean():.4f})')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title('Precision-Recall Curve - CatBoost Full Model', fontsize=14, fontweight='bold')
    plt.legend(loc="lower left", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "pr_curve.png"), dpi=300)
    print(f"PR curve salvata in: {os.path.join(PLOTS_DIR, 'pr_curve.png')}")
    plt.close()
    
    # ====== PLOT CONFUSION MATRIX ======
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Non-TD', 'TD'], yticklabels=['Non-TD', 'TD'])
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix - CatBoost Full Model', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "confusion_matrix.png"), dpi=300)
    print(f"Confusion matrix salvata in: {os.path.join(PLOTS_DIR, 'confusion_matrix.png')}")
    plt.close()
    
    # ====== PLOT DISTRIBUTION OF PREDICTIONS ======
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribuzione per classe vera
    ax1.hist(p_hat[y_true == 0], bins=50, alpha=0.6, label='Non-TD (0)', color='blue')
    ax1.hist(p_hat[y_true == 1], bins=50, alpha=0.6, label='TD (1)', color='red')
    ax1.axvline(best_thr, color='green', linestyle='--', linewidth=2, label=f'Soglia ottimale ({best_thr:.3f})')
    ax1.set_xlabel('Predicted Probability', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title('Distribution of Predicted Probabilities', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    
    # Logit distribution
    ax2.hist(pred_logit[y_true == 0], bins=50, alpha=0.6, label='Non-TD (0)', color='blue')
    ax2.hist(pred_logit[y_true == 1], bins=50, alpha=0.6, label='TD (1)', color='red')
    ax2.set_xlabel('Predicted Logit', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Distribution of Predicted Logits', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "prediction_distributions.png"), dpi=300)
    print(f"Distribution plots salvate in: {os.path.join(PLOTS_DIR, 'prediction_distributions.png')}")
    plt.close()
    
else:
    print("Label non presente -> salto metriche classificazione.")

# ====== FEATURE IMPORTANCE ======
try:
    feature_importance = model.get_feature_importance(train_pool)
    feature_names = feature_cols
    
    if len(feature_importance) > 0:
        # Crea dataframe con importanze
        fi_df = pd.DataFrame({
            'feature': feature_names[:len(feature_importance)],
            'importance': feature_importance
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 20 Feature Importance:")
        print(fi_df.head(20).to_string(index=False))
        
        # Plot top features
        top_n = min(30, len(fi_df))
        plt.figure(figsize=(10, max(8, top_n * 0.25)))
        plt.barh(range(top_n), fi_df['importance'].head(top_n), color='steelblue')
        plt.yticks(range(top_n), fi_df['feature'].head(top_n), fontsize=9)
        plt.xlabel('Importance', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.title(f'Top {top_n} Feature Importance - Full Model', fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "feature_importance.png"), dpi=300)
        print(f"\nFeature importance plot salvato in: {os.path.join(PLOTS_DIR, 'feature_importance.png')}")
        plt.close()
        
        # Salva feature importance in CSV
        fi_df.to_csv(os.path.join(PLOTS_DIR, "feature_importance.csv"), index=False)
        print(f"Feature importance CSV salvato in: {os.path.join(PLOTS_DIR, 'feature_importance.csv')}")
        
        # ====== EXTRA: Feature Importance by Type ======
        fi_df['type'] = fi_df['feature'].apply(
            lambda x: 'text' if x == TEXT_COL 
            else ('keyword' if x.startswith('kw_') 
                  else ('percentage' if x.startswith('pct_') 
                        else ('count' if x.startswith('n_') else 'boolean')))
        )
        
        type_importance = fi_df.groupby('type')['importance'].sum().sort_values(ascending=False)
        print("\nImportanza aggregata per tipo di feature:")
        print(type_importance)
        
        # Plot importanza per tipo
        plt.figure(figsize=(10, 6))
        type_importance.plot(kind='bar', color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
        plt.xlabel('Tipo Feature', fontsize=12)
        plt.ylabel('Importanza Totale', fontsize=12)
        plt.title('Importanza Aggregata per Tipo di Feature', fontsize=14, fontweight='bold')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "importance_by_type.png"), dpi=300)
        print(f"Type importance plot salvato in: {os.path.join(PLOTS_DIR, 'importance_by_type.png')}")
        plt.close()
        
except Exception as e:
    print(f"\nNote: Impossibile calcolare feature importance: {e}")

# ====== PLOT WEIGHTS DISTRIBUTION (se attivi) ======
if USE_BORDERLINE_WEIGHTS and train_w is not None:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.hist(train_w, bins=50, color='coral', edgecolor='black', alpha=0.7)
    plt.xlabel('Weight', fontsize=11)
    plt.ylabel('Frequency', fontsize=11)
    plt.title('Distribution of Training Weights', fontsize=12, fontweight='bold')
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    logits = train_df[TARGET_COL].values
    plt.scatter(logits, train_w, alpha=0.3, s=10, color='coral')
    plt.xlabel('Logit Value', fontsize=11)
    plt.ylabel('Weight', fontsize=11)
    plt.title('Weights vs Logit (Borderline Focus)', fontsize=12, fontweight='bold')
    plt.axvline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "weights_distribution.png"), dpi=300)
    print(f"Weights distribution plot salvato in: {os.path.join(PLOTS_DIR, 'weights_distribution.png')}")
    plt.close()

# ====== SAVE MODEL ======
model.save_model(MODEL_OUT)

# ====== SAVE METADATA ======
meta = {
    "model_type": "full",
    "description": "Modello completo con testo + cheap features + borderline weighting",
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df_feats.columns else None,
    "num_cols": num_cols,
    "num_numeric_features": len(num_cols),
    "feature_cols": feature_cols,
    "num_total_features": len(feature_cols),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "use_borderline_weights": USE_BORDERLINE_WEIGHTS,
    "w_max": W_MAX if USE_BORDERLINE_WEIGHTS else None,
    "dataset_size": len(df_feats),
    "train_size": len(train_df),
    "valid_size": len(valid_df),
    "rmse_train_logit": float(rmse_train),
    "rmse_valid_logit": float(rmse),
    "training_time_seconds": training_time,
    "training_time_minutes": training_time / 60,
    "co2_emissions_kg": float(emissions),
    "model_iterations": model.get_best_iteration(),
    "model_file": MODEL_OUT,
    **metrics
}

with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print("RIEPILOGO FINALE - MODELLO FULL")
print("="*60)
print(f"Modello salvato: {MODEL_OUT}")
print(f"Metadata salvati: {META_OUT}")
print(f"Grafici salvati in: {PLOTS_DIR}/")
print(f"Numero features: {len(feature_cols)} (1 testo + {len(num_cols)} numeriche)")
print(f"Borderline weighting: {'SI' if USE_BORDERLINE_WEIGHTS else 'NO'}")
print(f"Tempo training: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"CO2 emissions: {emissions:.6f} kg")
print(f"RMSE validation: {rmse:.6f}")
if metrics:
    print(f"ROC-AUC validation: {metrics.get('roc_auc_valid', 'N/A'):.6f}")
    print(f"PR-AUC validation: {metrics.get('pr_auc_valid', 'N/A'):.6f}")
    print(f"F1-Score validation: {metrics.get('f1_score', 'N/A'):.6f}")
print("="*60)


Caricamento dataset...
Dataset caricato: 20828 righe
Estrazione cheap features...


[codecarbon WARNING @ 20:34:06] Multiple instances of codecarbon are allowed to run at the same time.


Features estratte: 30 (1 testo + 29 numeriche)
Split train/validation...
Train: 16662 | Validation: 4166
Borderline weighting attivo: W_MAX=5.0, peso medio=1.103

Inizio training con carbon tracking...


[codecarbon INFO @ 20:34:08] [setup] RAM Tracking...
[codecarbon INFO @ 20:34:08] [setup] CPU Tracking...
[codecarbon WARNING @ 20:34:09] We saw that you have a 13th Gen Intel(R) Core(TM) i7-13620H but we don't know it. Please contact us.
[codecarbon WARNING @ 20:34:09] We will use the default power consumption of 4 W per thread for your 16 CPU, so 64W.
[codecarbon WARNING @ 20:34:09] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 20:34:09] CPU Model on constant consumption mode: 13th Gen Intel(R) Core(TM) i7-13620H
[codecarbon WARNING @ 20:34:09] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 20:34:09] [setup] GPU Tracking...
[codecarbon INFO @ 20:34:09] No GPU found.
[codecarbon INFO @ 20:34:09] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Me

0:	learn: 3.9763845	test: 4.0427264	best: 4.0427264 (0)	total: 284ms	remaining: 28m 22s


...

3800:	learn: 1.2434046	test: 2.4779881	best: 2.4775578 (3695)	total: 8m 14s	remaining: 4m 46s


[codecarbon INFO @ 20:42:30] Delta energy consumed for CPU with cpu_load : 0.000141 kWh, power : 34.9975198396 W
[codecarbon INFO @ 20:42:30] Energy consumed for All CPU : 0.004684 kWh
[codecarbon INFO @ 20:42:30] 0.006018 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 20:42:42] Energy consumed for RAM : 0.001368 kWh. RAM Power : 10.0 W


Stopped by overfitting detector  (200 iterations wait)

bestTest = 2.477557756
bestIteration = 3695

Shrink model to first 3696 iterations.


[codecarbon INFO @ 20:42:43] Delta energy consumed for CPU with cpu_load : 0.000113 kWh, power : 32.55189047817143 W
[codecarbon INFO @ 20:42:43] Energy consumed for All CPU : 0.004798 kWh
[codecarbon INFO @ 20:42:43] 0.006166 kWh of electricity and 0.000000 L of water were used since the beginning.



Training completato in 508.68 secondi (8.48 minuti)
Emissioni CO2: 0.002039 kg

Generazione predizioni...

RMSE(logit) train: 1.268272
RMSE(logit) valid: 2.477558

ROC-AUC train: 0.985986
ROC-AUC valid: 0.940778
PR-AUC train: 0.986669
PR-AUC valid: 0.943789

Soglia ottimale (F1): 0.319261
Accuracy: 0.861018
Precision: 0.842491
Recall: 0.886747
F1-Score: 0.864053

Confusion Matrix:
[[1747  344]
 [ 235 1840]]

ROC curve salvata in: plots_catboost_full\roc_curve.png
PR curve salvata in: plots_catboost_full\pr_curve.png
Confusion matrix salvata in: plots_catboost_full\confusion_matrix.png
Distribution plots salvate in: plots_catboost_full\prediction_distributions.png

Top 20 Feature Importance:
          feature  importance
        orig_text   88.741957
          n_words    3.993729
          n_chars    3.845895
      kw_debt_cnt    0.970982
  kw_refactor_cnt    0.660790
       has_should    0.550907
      kw_debt_has    0.488049
  kw_refactor_has    0.361761
        pct_punct    0.331817